# 04 — Feature Importance & Análisis SHAP

¿Por qué predice bien el modelo XGBoost? Este notebook responde esa pregunta con dos enfoques:

1. **Feature importance nativo** de XGBoost (gain promedio por feature)
2. **SHAP values** — explicabilidad global e individual del modelo
3. **Análisis de errores** — ¿en qué horas y días falla más el modelo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

## 1. Cargar Datos y Construir Features

In [ ]:
df = pd.read_csv("../data/raw/AEP_hourly.csv")
df.columns = ["datetime", "aep_mw"]
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

# Eliminar duplicados (igual que en el pipeline)
df = df.groupby("datetime", as_index=False)["aep_mw"].mean()
full_range = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="h")
df = df.set_index("datetime").reindex(full_range)
df.index.name = "datetime"
df["aep_mw"] = df["aep_mw"].interpolate(method="time")
df = df.reset_index()

# Features de calendario
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# Lags
df["lag_1"]   = df["aep_mw"].shift(1)
df["lag_24"]  = df["aep_mw"].shift(24)
df["lag_168"] = df["aep_mw"].shift(168)

# Rolling stats
df["rolling_24"]  = df["aep_mw"].shift(1).rolling(24).mean()
df["rolling_168"] = df["aep_mw"].shift(1).rolling(168).mean()

df = df.dropna().reset_index(drop=True)
print(f"Dataset final: {len(df):,} filas")
df.head()

## 2. Entrenar XGBoost

Split temporal: 80% train, 20% test — sin shuffle para respetar el orden temporal.

In [ ]:
FEATURES = ["hour", "dayofweek", "is_weekend",
            "lag_1", "lag_24", "lag_168",
            "rolling_24", "rolling_168"]
TARGET = "aep_mw"

split = int(len(df) * 0.8)
train = df.iloc[:split]
test  = df.iloc[split:]

model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42
)
model.fit(train[FEATURES], train[TARGET])

test = test.copy()
test["yhat"] = model.predict(test[FEATURES])
test["error"] = test["yhat"] - test[TARGET]
test["abs_error"] = test["error"].abs()

wape = test["abs_error"].sum() / test[TARGET].sum() * 100
mae  = test["abs_error"].mean()
print(f"Test WAPE: {wape:.2f}%")
print(f"Test MAE:  {mae:.1f} MW")

## 3. Feature Importance (Nativo XGBoost)

In [ ]:
feat_imp = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["tomato" if v == feat_imp["importance"].max() else "steelblue" 
          for v in feat_imp["importance"]]
ax.barh(feat_imp["feature"], feat_imp["importance"], color=colors, edgecolor="white")
ax.set_title("Feature Importance — XGBoost (F-score gain)", fontsize=13)
ax.set_xlabel("Importancia relativa")
for i, (val, name) in enumerate(zip(feat_imp["importance"], feat_imp["feature"])):
    ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Análisis SHAP

SHAP explica la **contribución de cada feature a cada predicción individual**, no solo el promedio global.

In [ ]:
# Calcular SHAP values sobre el set de test (sample de 2000 para velocidad)
sample = test[FEATURES].sample(2000, random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

shap_df = pd.DataFrame(shap_values, columns=FEATURES)
print("SHAP values calculados. Shape:", shap_df.shape)

In [ ]:
# Summary plot — importancia global + distribución de impacto
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample, plot_type="bar", show=False)
plt.title("SHAP — Importancia Global de Features", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot — impacto y dirección por feature
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample, show=False)
plt.title("SHAP Beeswarm — Dirección e Intensidad del Impacto", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Análisis de Errores

¿En qué horas y días el modelo falla más?

In [ ]:
# Error absoluto por hora del día
error_hour = test.groupby("hour")["abs_error"].mean()

fig, ax = plt.subplots(figsize=(13, 4))
colors = ["tomato" if v == error_hour.max() else "steelblue" for v in error_hour]
ax.bar(error_hour.index, error_hour.values, color=colors, edgecolor="white")
ax.set_title("Error Absoluto Promedio por Hora del Día", fontsize=13)
ax.set_xlabel("Hora")
ax.set_ylabel("MAE (MW)")
ax.set_xticks(range(24))
ax.axhline(error_hour.mean(), color="gray", linestyle="--", linewidth=1, label=f"Promedio: {error_hour.mean():.0f} MW")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Error absoluto por día de la semana
day_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
error_dow = test.groupby("dayofweek")["abs_error"].mean()

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ["tomato"] * 5 + ["steelblue"] * 2
ax.bar(range(7), error_dow.values, color=bar_colors, edgecolor="white")
ax.set_title("Error Absoluto Promedio por Día de la Semana", fontsize=13)
ax.set_xlabel("Día")
ax.set_ylabel("MAE (MW)")
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.axhline(error_dow.mean(), color="gray", linestyle="--", linewidth=1, label=f"Promedio: {error_dow.mean():.0f} MW")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Real vs Predicho — muestra de 2 semanas
sample_plot = test.iloc[:24*14]

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(sample_plot["datetime"], sample_plot[TARGET], label="Real", color="steelblue", linewidth=1.5)
ax.plot(sample_plot["datetime"], sample_plot["yhat"], label="Predicho", color="tomato",
        linewidth=1.5, linestyle="--")
ax.set_title("Real vs Predicho — Muestra de 2 Semanas", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

## Conclusiones

| Hallazgo | Interpretación |
|---|---|
| `lag_1` y `lag_24` dominan la importancia | El consumo de hace 1h y 24h son los predictores más fuertes |
| `rolling_24` aporta más que `rolling_168` | La media reciente captura mejor el estado actual que la media semanal |
| `hour` supera a `dayofweek` | La hora del día explica más varianza que el día en sí |
| Mayor error en horas pico (17-20h) | Los picos vespertinos son más difíciles de predecir |
| Lunes tiene mayor error que fin de semana | El cambio lunes/fin de semana genera más incertidumbre |